# ResearchLanka Kaggle Main-Branch Full Run

Import this notebook into Kaggle and run cells from top to bottom.

It will:

- clone/pull the latest `main` branch
- copy your uploaded raw dataset into the repo
- install Python + Dagster dependencies
- run the Dagster no-collection preprocessing job
- build best-quality embeddings
- train Logistic Regression
- train Linear SVM
- compare model metrics
- zip outputs for download

It does **not** collect data from APIs or repositories.

## 1. Settings

In [1]:
from pathlib import Path

REPO_URL = "https://github.com/krish-anu/researchlanka-ai.git"
BRANCH = "main"
WORK_DIR = Path("/kaggle/working")
CODE_DIR = WORK_DIR / "code"
BACKEND_DIR = CODE_DIR / "backend"
DATASET_DATA_DIR = Path("/kaggle/input/datasets/anusankrishnathas/raw-data1/backend/data")
OUTPUT_ZIP = WORK_DIR / "researchlanka-kaggle-outputs.zip"

print("Repo:", REPO_URL)
print("Branch:", BRANCH)
print("Code dir:", CODE_DIR)
print("Backend dir:", BACKEND_DIR)
print("Dataset data dir:", DATASET_DATA_DIR)
print("Output zip:", OUTPUT_ZIP)

Repo: https://github.com/krish-anu/researchlanka-ai.git
Branch: main
Code dir: /kaggle/working/code
Backend dir: /kaggle/working/code/backend
Dataset data dir: /kaggle/input/datasets/anusankrishnathas/raw-data1/backend/data
Output zip: /kaggle/working/researchlanka-kaggle-outputs.zip


## 2. Check Kaggle Dataset Exists

If this fails, your Kaggle dataset path is different. Update `DATASET_DATA_DIR` above.

In [2]:
!ls -la /kaggle/input
!find /kaggle/input -maxdepth 5 -type d | head -80
!test -d {DATASET_DATA_DIR} && echo "Dataset path OK" || echo "Dataset path NOT FOUND"

total 12
drwxr-xr-x 3 root root 4096 Aug 28 03:25 .
drwxr-xr-x 8 root root 4096 Aug 28 03:25 ..
drwxr-xr-x 3 root root 4096 Aug 28 03:25 datasets
/kaggle/input
/kaggle/input/datasets
/kaggle/input/datasets/anusankrishnathas
/kaggle/input/datasets/anusankrishnathas/raw-data1
/kaggle/input/datasets/anusankrishnathas/raw-data1/backend
/kaggle/input/datasets/anusankrishnathas/raw-data1/backend/data
Dataset path OK


## 3. Clone Or Pull Latest Main Branch

In [3]:
%cd /kaggle/working
if CODE_DIR.exists() and (CODE_DIR / ".git").exists():
    %cd /kaggle/working/code
    !git fetch origin {BRANCH}
    !git checkout {BRANCH}
    !git pull origin {BRANCH}
else:
    !git clone -b {BRANCH} {REPO_URL} code
    %cd /kaggle/working/code

!git log --oneline -3
!ls

/kaggle/working
Cloning into 'code'...
remote: Enumerating objects: 3234, done.
remote: Counting objects: 100% (447/447), done.
remote: Compressing objects: 100% (359/359), done.
remote: Total 3234 (delta 155), reused 151 (delta 80), pack-reused 2787 (from 2)
Receiving objects: 100% (3234/3234), 13.44 MiB | 14.88 MiB/s, done.
Resolving deltas: 100% (1877/1877), done.
/kaggle/working/code
fffe167 (HEAD -> main, origin/main, origin/HEAD) Merge pull request #469 from krish-anu/test-fixing
33c6751 (origin/test-fixing) Merge branch 'main' into test-fixing
e9f0619 Merge pull request #473 from krish-anu/feature/openalex-full-data-collection
backend       dse-project.ipynb  KAGGLE_README.md  notebooks
CHANGELOG.md  frontend		 Makefile	   README.md


## 4. Copy Uploaded Raw Data Into Backend

In [4]:
%cd /kaggle/working/code/backend
!rm -rf data
!mkdir -p data
!cp -r {DATASET_DATA_DIR}/* data/
!find data -maxdepth 3 -type f | head -60

/kaggle/working/code/backend
data/config/repositories.json
data/processed/crossref/crossref_sri_lanka_works.csv
data/processed/crossref/crossref_sri_lanka_works.jsonl
data/processed/repositories/jfn_medicine.jsonl
data/processed/repositories/busl.jsonl
data/processed/repositories/ou.jsonl
data/processed/repositories/sliit.jsonl
data/processed/repositories/uom.jsonl
data/processed/repositories/jfn_research.jsonl
data/processed/repositories/pdn.jsonl
data/processed/repositories/cmb.jsonl
data/processed/repositories/ruh.jsonl
data/processed/repositories/seu.jsonl
data/processed/repositories/nsf.jsonl
data/processed/repositories_combined.csv
data/processed/sljol.csv
data/reports/dagster_collection_summary_20260804T085133Z.json
data/raw/uom/oai_dc.jsonl
data/raw/ou/oai_dc.jsonl
data/raw/vau/oai_dc.jsonl
data/raw/cmb/rest_items.jsonl
data/raw/rjt/oai_dc.jsonl
data/raw/esn/oai_dc.jsonl
data/raw/seu/oai_dc.jsonl
data/raw/sltc/oai_dc.jsonl
data/raw/jfn_medicine/html_meta.jsonl
data/raw/openalex

## 5. Install Dependencies

Kaggle may show dependency conflict warnings. Continue if the install completes. If the session restarts, rerun from the top.

In [5]:
%cd /kaggle/working/code/backend
!pip install $(grep -v '^psycopg2==' requirements.txt)
!pip install dagster==1.13.16 dagster-webserver
!pip install -e dagster-quickstart
!python -m dagster --version

/kaggle/working/code/backend
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.7/41.7 kB 1.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 133.3/133.3 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 224.3/224.3 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.4/117.4 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.5/65.5 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 68.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.2/100.2 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 79.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.7/47.7 MB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 386.5/386.5 kB 460.5 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━

## 6. Run Dagster Pipeline Without Data Collection

This prepares existing source files and runs preprocessing through the analysis-ready dataset.

In [6]:
%cd /kaggle/working/code/backend/dagster-quickstart
!python -m dagster job execute \
  -m dagster_quickstart.definitions \
  -j researchlanka_no_collection_preprocessing_job

/kaggle/working/code/backend/dagster-quickstart
/usr/local/lib/python3.12/dist-packages/click/core.py:853: SupersessionWarning: Function `job_execute_command` is superseded and its usage is discouraged. Use 'dg launch --job <job_name>' instead.
  return callback(*args, **kwargs)

  Telemetry:

  As an open-source project, we collect usage statistics to inform development priorities. For more
  information, read https://docs.dagster.io/about/telemetry.

  We will not see or store any data that is processed by your code.

  To opt-out, add the following to $DAGSTER_HOME/dagster.yaml, creating that file if necessary:

    telemetry:
      enabled: false


  Welcome to Dagster!

  If you have any questions or would like to engage with the Dagster team, please join us on Slack
  (https://bit.ly/39dvSsF).

2026-08-28 03:27:47 +0000 - dagster - DEBUG - researchlanka_no_collection_preprocessing_job - a67773ab-3172-4178-af0a-5b1ee4735c46 - 126 - RUN_START - Started execution of run for "researc

## 7. Verify Preprocessing Outputs

In [7]:
%cd /kaggle/working/code/backend
!ls -lh data/processed/repositories_combined.csv
!ls -lh data/processed/sljol.csv
!ls -lh data/processed/common/common_publications_final.csv
!ls -lh data/processed/common/common_publications_final_2016_2026_analysis_ready.csv

import pandas as pd
paths = [
    'data/processed/common/common_publications_final.csv',
    'data/processed/common/common_publications_final_2016_2026_analysis_ready.csv',
]
for path in paths:
    frame = pd.read_csv(path, nrows=5)
    total = sum(1 for _ in open(path, encoding='utf-8')) - 1
    print(path, 'rows=', total, 'columns=', len(frame.columns))


/kaggle/working/code/backend
-rw-r--r-- 1 root root 126M Aug 28 03:28 data/processed/repositories_combined.csv
-rw-r--r-- 1 root root 49M Aug 28 03:28 data/processed/sljol.csv
-rw-r--r-- 1 root root 286M Aug 28 03:46 data/processed/common/common_publications_final.csv
-rw-r--r-- 1 root root 377M Aug 28 03:54 data/processed/common/common_publications_final_2016_2026_analysis_ready.csv
data/processed/common/common_publications_final.csv rows= 178162 columns= 56
data/processed/common/common_publications_final_2016_2026_analysis_ready.csv rows= 137856 columns= 71


## 8. Build Best-Quality Embeddings

Full text fields, trigrams, larger vocabulary, 512 dimensions, no row limit.

In [8]:
%cd /kaggle/working/code/backend
!make model-embeddings PYTHON=python \
  EMBED_TEXT_COLUMNS=title,abstract,topics,keywords,concepts \
  EMBED_MAX_FEATURES=100000 \
  EMBED_NGRAM_MAX=3 \
  EMBED_DIM=512
!ls -lh data/models/publication_text_embeddings.parquet data/models/publication_text_embedding_model.joblib data/models/publication_text_embeddings_summary.txt

/kaggle/working/code/backend
python scripts/modeling/generate_publication_text_embeddings.py --input data/processed/common/common_publications_final.csv --output data/models/publication_text_embeddings.parquet --model-output data/models/publication_text_embedding_model.joblib --manifest-output data/models/publication_text_embeddings_manifest.json --summary-output data/models/publication_text_embeddings_summary.txt --text-columns title,abstract,topics,keywords,concepts --metadata-columns record_number,publication_year,title,doi,openalex_id,source_dataset,source_institution_id,source_record_id --embedding-dim 512 --max-features 100000 --min-df 2 --max-df 0.95 --ngram-max 3 
Generated publication text embeddings: rows=178152, dim=512, output=data/models/publication_text_embeddings.parquet
-rw------- 1 root root 395M Aug 28 04:02 data/models/publication_text_embedding_model.joblib
-rw-r--r-- 1 root root 548M Aug 28 04:02 data/models/publication_text_embeddings.parquet
-rw------- 1 root roo

## 9. Train Best-Quality Logistic Regression

In [9]:
%cd /kaggle/working/code/backend
!make train-logreg PYTHON=python \
  LOGREG_TEXT_COLUMNS=title,abstract,topics,keywords,concepts \
  LOGREG_MAX_FEATURES=100000 \
  LOGREG_NGRAM_MAX=3 \
  LOGREG_MAX_ITER=2000
!cat data/models/logistic_regression_primary_domain_metrics.txt

/kaggle/working/code/backend
python scripts/modeling/train_logistic_regression_classifier.py --input data/processed/common/common_publications_final.csv --label-column primary_domain --text-columns title,abstract,topics,keywords,concepts --model-output data/models/logistic_regression_primary_domain.joblib --metrics-output data/models/logistic_regression_primary_domain_metrics.txt --label-counts-output data/models/logistic_regression_primary_domain_labels.csv --predictions-output data/models/logistic_regression_primary_domain_predictions.csv --manifest-output data/models/logistic_regression_primary_domain_manifest.json --max-features 100000 --min-df 2 --max-df 0.95 --ngram-max 3 --min-class-count 20 --test-size 0.2 --max-iter 2000 
Trained logistic_regression classifier on 52,268 rows.
Classes: 4
Accuracy: 0.8704
Balanced accuracy: 0.8655
Macro F1: 0.8620
Model: data/models/logistic_regression_primary_domain.joblib
Model SHA-256: 47f556547d2ec3f1f39187b21fb79734832a695c0d6832ac7b1a803c7

In [10]:
%cd /kaggle/working/code/backend
!pip install -e . --no-deps

/kaggle/working/code/backend
Obtaining file:///kaggle/working/code/backend
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for research-analytics-framework (pyproject.toml) ... done
  Created wheel for research-analytics-framework: filename=research_analytics_framework-0.1.0-0.editable-py3-none-any.whl size=8173 sha256=9d83522153caca43db03c13acf632201ec739b2011b18fc80049ee94bebed11f
  Stored in directory: /tmp/pip-ephem-wheel-cache-sse45hwn/wheels/48/6f/98/3205cf08ddaa25af658fb7d96745a6be29b5c675f81653f651
Successfully built research-analytics-framework


## 10. Train Best-Quality Linear SVM

If Kaggle RAM fails, rerun this cell with `--max-features 50000`.

In [11]:
%cd /kaggle/working/code/backend
!python scripts/modeling/train_linear_svm_classifier.py \
  --input data/processed/common/common_publications_final.csv \
  --label-column primary_domain \
  --text-columns title,abstract,topics,keywords,concepts \
  --ngram-max 3 \
  --max-features 100000 \
  --c-values 0.1,1,10 \
  --cv-folds 3 \
  --class-weight balanced \
  --max-iter 5000
!cat data/models/linear_svm_primary_domain_metrics.txt

/kaggle/working/code/backend
Traceback (most recent call last):
  File "/kaggle/working/code/backend/scripts/modeling/train_linear_svm_classifier.py", line 10, in <module>
    from src.modeling.linear_svm_training import main
  File "/kaggle/working/code/backend/src/modeling/linear_svm_training.py", line 43, in <module>
    from src.preprocessing.text_cleaning import (
  File "/kaggle/working/code/backend/src/preprocessing/text_cleaning.py", line 147
    <<<<<<< HEAD
    ^^
SyntaxError: invalid syntax
cat: data/models/linear_svm_primary_domain_metrics.txt: No such file or directory


## 11. Compare Models

In [12]:
%cd /kaggle/working/code/backend
!grep -E "model_family|label_column|accuracy|macro_f1|weighted_f1" data/models/*metrics.txt || true

/kaggle/working/code/backend
model_family: logistic_regression
label_column: primary_domain
accuracy: 0.8704
macro_f1: 0.8620
weighted_f1: 0.8707
         accuracy                           0.87     10454
accuracy: 0.8704
balanced_accuracy: 0.8655
macro_f1: 0.8620
weighted_f1: 0.8707


## 12. Zip Outputs For Download

In [13]:
%cd /kaggle/working/code/backend
!rm -f /kaggle/working/researchlanka-kaggle-outputs.zip
!zip -r /kaggle/working/researchlanka-kaggle-outputs.zip data/processed data/models
!ls -lh /kaggle/working/researchlanka-kaggle-outputs.zip

/kaggle/working/code/backend
  adding: data/processed/ (stored 0%)
  adding: data/processed/common/ (stored 0%)
  adding: data/processed/common/common_publications_final_2016_2026_multivalue_normalized.csv (deflated 70%)
  adding: data/processed/common/publication_references.csv (deflated 85%)
  adding: data/processed/common/common_publications_deduplicated.csv (deflated 74%)
  adding: data/processed/common/common_publications_final_2016_2026.csv (deflated 70%)
  adding: data/processed/common/publication_multivalue_items_2016_2026.csv (deflated 90%)
  adding: data/processed/common/common_publications_final_2016_2026_language_normalized_summary.csv (deflated 50%)
  adding: data/processed/common/common_publications_final.csv (deflated 70%)
  adding: data/processed/common/common_publications_final_2016_2026_analysis_ready_summary.csv (deflated 62%)
  adding: data/processed/common/common_publications_final_2016_2026_language_normalized_mapping.csv (deflated 45%)
  adding: data/processed/co

Download this file from the Kaggle output panel:

```text
/kaggle/working/researchlanka-kaggle-outputs.zip
```

In [14]:
import os

for name in os.listdir("/kaggle/working"):
    path = os.path.join("/kaggle/working", name)
    size_gb = os.path.getsize(path) / (1024**3) if os.path.isfile(path) else 0
    kind = "file" if os.path.isfile(path) else "folder"
    print(kind, name, round(size_gb, 3), "GB")

file __notebook__.ipynb 0.0 GB
file researchlanka-kaggle-outputs.zip 2.008 GB
folder code 0 GB


In [15]:
import os

source_file = "/kaggle/working/researchlanka-kaggle-outputs.zip"
output_dir = "/kaggle/working/split_outputs"
os.makedirs(output_dir, exist_ok=True)

chunk_size = 500 * 1024 * 1024

with open(source_file, "rb") as f:
    part = 1
    while True:
        chunk = f.read(chunk_size)
        if not chunk:
            break

        part_path = os.path.join(output_dir, f"researchlanka_outputs_part_{part:03d}.zip.part")
        with open(part_path, "wb") as out:
            out.write(chunk)

        print("Created:", part_path, round(os.path.getsize(part_path) / (1024**2), 1), "MB")
        part += 1

Created: /kaggle/working/split_outputs/researchlanka_outputs_part_001.zip.part 500.0 MB
Created: /kaggle/working/split_outputs/researchlanka_outputs_part_002.zip.part 500.0 MB
Created: /kaggle/working/split_outputs/researchlanka_outputs_part_003.zip.part 500.0 MB
Created: /kaggle/working/split_outputs/researchlanka_outputs_part_004.zip.part 500.0 MB
Created: /kaggle/working/split_outputs/researchlanka_outputs_part_005.zip.part 56.3 MB


In [16]:
import os
import zipfile
import glob

# Find model folders
for path in glob.glob("/kaggle/working/**/data/models", recursive=True):
    print("FOUND:", path)

FOUND: /kaggle/working/code/backend/data/models


In [17]:
import zipfile
import os

source_zip = "/kaggle/working/researchlanka-kaggle-outputs.zip"
models_zip = "/kaggle/working/researchlanka-models-only.zip"

with zipfile.ZipFile(source_zip, "r") as src:
    names = src.namelist()
    
    model_files = [
        name for name in names
        if "/data/models/" in name or name.startswith("data/models/")
    ]
    
    print("Model files found:", len(model_files))
    for name in model_files[:20]:
        print(name)

    with zipfile.ZipFile(models_zip, "w", zipfile.ZIP_DEFLATED) as dst:
        for name in model_files:
            dst.writestr(name, src.read(name))

print("Created:", models_zip)
print("Size MB:", round(os.path.getsize(models_zip) / (1024**2), 2))

Model files found: 12
data/models/
data/models/logistic_regression_primary_domain_per_class.csv
data/models/logistic_regression_primary_domain_labels.csv
data/models/logistic_regression_primary_domain_manifest.json
data/models/publication_text_embeddings_manifest.json
data/models/publication_text_embedding_model.joblib
data/models/logistic_regression_primary_domain_confusion_matrix.csv
data/models/logistic_regression_primary_domain_metrics.txt
data/models/publication_text_embeddings_summary.txt
data/models/logistic_regression_primary_domain_predictions.csv
data/models/logistic_regression_primary_domain.joblib
data/models/publication_text_embeddings.parquet
Created: /kaggle/working/researchlanka-models-only.zip
Size MB: 887.28


In [18]:
from IPython.display import FileLink, display

display(FileLink("/kaggle/working/researchlanka-models-only.zip"))

/kaggle/working/researchlanka-models-only.zip

In [19]:
import zipfile

source_zip = "/kaggle/working/researchlanka-kaggle-outputs.zip"

with zipfile.ZipFile(source_zip, "r") as z:
    for name in z.namelist()[:100]:
        print(name)

data/processed/
data/processed/common/
data/processed/common/common_publications_final_2016_2026_multivalue_normalized.csv
data/processed/common/publication_references.csv
data/processed/common/common_publications_deduplicated.csv
data/processed/common/common_publications_final_2016_2026.csv
data/processed/common/publication_multivalue_items_2016_2026.csv
data/processed/common/common_publications_final_2016_2026_language_normalized_summary.csv
data/processed/common/common_publications_final.csv
data/processed/common/common_publications_final_2016_2026_analysis_ready_summary.csv
data/processed/common/common_publications_final_2016_2026_language_normalized_mapping.csv
data/processed/common/publication_count_audit.csv
data/processed/common/common_publications_deduplicated_stream_summary.csv
data/processed/common/common_publications_final_2016_2026_summary.csv
data/processed/common/common_publications_final_summary.csv
data/processed/common/common_publications_final_2016_2026_multivalue_no

In [20]:
import zipfile
import os

source_zip = "/kaggle/working/researchlanka-kaggle-outputs.zip"
models_zip = "/kaggle/working/researchlanka-models-only.zip"

with zipfile.ZipFile(source_zip, "r") as src:
    model_files = [
        name for name in src.namelist()
        if name.startswith("data/models/") and not name.endswith("/")
    ]

    print("Model files found:", len(model_files))

    with zipfile.ZipFile(models_zip, "w", zipfile.ZIP_DEFLATED) as dst:
        for name in model_files:
            dst.writestr(name, src.read(name))
            print("Added:", name)

print("Created:", models_zip)
print("Size MB:", round(os.path.getsize(models_zip) / (1024**2), 2))

Model files found: 11
Added: data/models/logistic_regression_primary_domain_per_class.csv
Added: data/models/logistic_regression_primary_domain_labels.csv
Added: data/models/logistic_regression_primary_domain_manifest.json
Added: data/models/publication_text_embeddings_manifest.json
Added: data/models/publication_text_embedding_model.joblib
Added: data/models/logistic_regression_primary_domain_confusion_matrix.csv
Added: data/models/logistic_regression_primary_domain_metrics.txt
Added: data/models/publication_text_embeddings_summary.txt
Added: data/models/logistic_regression_primary_domain_predictions.csv
Added: data/models/logistic_regression_primary_domain.joblib
Added: data/models/publication_text_embeddings.parquet
Created: /kaggle/working/researchlanka-models-only.zip
Size MB: 887.28


In [21]:
from IPython.display import FileLink, display

display(FileLink("/kaggle/working/researchlanka-models-only.zip"))

/kaggle/working/researchlanka-models-only.zip